In [1]:
from tools.Handler import *

In [2]:
import pandas as pd
import json

In [3]:
# ---- 5. (later) load it back ----
df_loaded = pd.read_csv(r"question_answers\questions_with_source.csv")
df_loaded["embedding"] = df_loaded["embedding"].apply(
    lambda v: json.loads(v) if isinstance(v, str) and v else None
)

In [4]:
print("Reloaded OK. First embedding length:", len(df_loaded["embedding"].iloc[0]))

Reloaded OK. First embedding length: 768


In [5]:
df_loaded = df_loaded.rename(columns={'ideal_answer': 'answer'})

In [5]:
df_loaded['experience']="common"

In [7]:
df_loaded['difficulty']="easy"

In [8]:
df_loaded.head()

,question,answer,keywords,role,categories,weights,embedding,experience,difficulty
0,Walk me through your resume,NaN,"[skills, portfolio, eductional backgrounds]",common,"['introduction', 'experience']","[{'label': 'introduction', 'score': 0.56}, {'l...","[0.0034455983, -0.04018085, -0.15762876, -0.01...",common,easy


In [9]:
source_name="https://www.themuse.com/advice/interview-questions-and-answers"
source_description ="60+ most common job interview questions and answers"

#### old one

In [10]:
import ast

def to_json_text(value):
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return "null"
        try:
            value = json.loads(value)
        except json.JSONDecodeError:
            value = ast.literal_eval(value)

    return json.dumps(value, ensure_ascii=False)

df_to_insert = df_loaded.copy()

for column in ["keywords", "categories", "weights", "embedding"]:
    if column in df_to_insert.columns:
        df_to_insert[column] = df_to_insert[column].apply(to_json_text)

batch_insert_questions(
    df_to_insert,
    source_name=source_name,
    source_description=source_description
)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (<unknown>, line 1)

#### old one

#### new one

In [11]:
import ast
import json

def to_json_text(value):
    if value is None:
        return "null"

    if isinstance(value, str):
        value = value.strip()
        if not value:
            return "null"

        try:
            parsed = json.loads(value)
        except (TypeError, json.JSONDecodeError):
            try:
                parsed = ast.literal_eval(value)
            except (ValueError, SyntaxError):
                parsed = value
        value = parsed

    elif hasattr(value, "tolist"):
        value = value.tolist()

    return json.dumps(value, ensure_ascii=False, default=str)

df_to_insert = df_loaded.copy()

for column in ["keywords", "categories", "weights", "embedding"]:
    if column in df_to_insert.columns:
        df_to_insert[column] = df_to_insert[column].apply(to_json_text)

batch_insert_questions(
    df_to_insert,
    source_name=source_name,
    source_description=source_description
)

📌 Using source_id: 4
📊 Total rows: 1
✅ Batch 1: 1 inserted (0–1)

🎉 Done. inserted=1, skipped=0, source_id=4


In [ ]:
# import pandas as pd

# # 1. Load CSV
# # df = pd.read_csv("your_file.csv")

# # 2. Check total rows and unique rows
# total_rows = len(df)
# unique_rows = df.drop_duplicates().shape[0]

# print(f"📊 Total rows: {total_rows}")
# print(f"🔹 Unique rows: {unique_rows}")
# print(f"🔸 Duplicate rows: {total_rows - unique_rows}")

# # 3. Show duplicate rows (if any)
# duplicates = df[df.duplicated(keep=False)]
# if not duplicates.empty:
#     print(f"\n⚠️ Found {len(duplicates)} duplicate rows (including original):")
#     print(duplicates.head(10))  # Show first 10 duplicates
# else:
#     print("\n✅ No duplicates found!")

In [2]:
from tools.context_handler import relevant_context

In [3]:
results = relevant_context("What is the capital of France?", top_k=3)

In [4]:
results

["Question: Imagine you've been tasked to migrate our custom warehouse management system from on-premises to AWS. The team lead insists on using Terraform for IaC, but you have concerns about its compatibility with our legacy codebase. Describe your approach to addressing this issue while maintaining open communication with the team and ensuring minimal disruption to our business operations.\nIdeal_Answer: This comes down to making the trade-offs explicit and then acting decisively. I'd start by getting clarity on priorities directly from stakeholders rather than assuming, then communicate trade-offs transparently to everyone affected, paying close attention to AWS, Terraform, IaC, legacy codebase since those are the constraints that actually drive the right answer here. Concretely, I'd lay out the options, flag the risks tied to AWS, and agree with stakeholders on which trade-off we're willing to accept before committing to a plan. This is the kind of situation where sticking to the f